# LAB 02 — ELT Ingestion: CSV, JSON, Parquet → Delta

**Databricks Fundamentals | ~60 min | Difficulty: ★★★☆☆**

---

## Scenario
You are ingesting raw data files from a shared Volume into Delta tables.
Your job is to read files in different formats, define schemas explicitly,
add audit metadata and write clean Bronze tables.

## Files used
| File | Format | Path (relative to DATASET_PATH) |
|------|--------|--------------------------------|
| customers.csv | CSV | `customers/customers.csv` |
| orders_batch.json | JSON | `orders/orders_batch.json` |
| products (parquet) | Parquet | auto-detected |

## Task overview
| # | Topic | Fill-in? |
|---|-------|----------|
| 01 | Setup & auto-schema CSV read | given |
| 02 | Manual StructType schema for CSV | **TODO** |
| 03 | Filter, cast, rename columns | **TODO** |
| 04 | JSON auto-inference read | given |
| 05 | Manual schema for JSON | **TODO** |
| 06 | Add audit metadata columns | **TODO** |
| 07 | Write Bronze Delta table (overwrite) | **TODO** |
| 08 | Verify table — DESCRIBE DETAIL | given |
| 09 | Append second batch & check idempotency | **TODO** |
| 10 | DataFrame transformations | **TODO** |
| 11 | inferSchema vs manual schema benchmark | given |
| 12 | Read Parquet → write Delta | **TODO** |


In [ ]:
%run ../setup/00_setup

In [ ]:
# Task 01 — Read CSV: auto-schema inference
# This is GIVEN — run it and observe the result

df_customers_infer = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(f"{DATASET_PATH}/customers/customers.csv")
)

print("=== Schema (inferred) ===")
df_customers_infer.printSchema()
df_customers_infer.show(5, truncate=False)

# ❓ Question: What types did Spark infer for date-like columns?
# ❓ Question: Is inferSchema safe for production pipelines? Why / why not?


In [ ]:
# Task 02 — Read CSV: manual StructType schema
# TODO: import the required types and define the schema explicitly

from pyspark.sql.types import (
    # TODO: import StructType, StructField and all needed column types
    # Hint: you need StringType, IntegerType, DateType, DoubleType
)

customer_schema = StructType([
    # TODO: define all fields matching the CSV header:
    # customer_id (String, not null), first_name (String), last_name (String),
    # email (String), country (String), registration_date (DateType),
    # loyalty_points (IntegerType), annual_spend (DoubleType)
])

df_customers = (
    spark.read
         .option("header", "true")
         .option("dateFormat", "yyyy-MM-dd")
         # TODO: pass the schema to the reader
         .csv(f"{DATASET_PATH}/customers/customers.csv")
)

print("=== Schema (manual) ===")
df_customers.printSchema()
df_customers.show(5, truncate=False)


In [ ]:
# Task 03 — Filter, cast and rename columns
# TODO: complete all three operations using the df_customers DataFrame

from pyspark.sql import functions as F

# 3a. Select only: customer_id, first_name, last_name, country, annual_spend
df_selected = df_customers.select(
    # TODO: list the columns
)

# 3b. Filter: keep only customers with annual_spend > 500
df_filtered = df_selected.filter(
    # TODO: write the condition
)

# 3c. Rename annual_spend → yearly_spend, add column full_name = first_name + " " + last_name
df_enriched = (
    df_filtered
    # TODO: rename annual_spend to yearly_spend
    # TODO: add withColumn full_name using concat_ws(" ", ...)
)

df_enriched.show(10, truncate=False)
print("Row count:", df_enriched.count())


In [ ]:
# Task 04 — Read JSON orders: auto-inference (GIVEN)

df_orders_infer = (
    spark.read
         .option("multiLine", "false")
         .json(f"{DATASET_PATH}/orders/orders_batch.json")
)

print("=== JSON inferred schema ===")
df_orders_infer.printSchema()
df_orders_infer.show(5, truncate=False)

# ❓ Question: What type does Spark assign to total_amount and unit_price?
# ❓ Question: What happened to order_datetime — is it a string or timestamp?


In [ ]:
# Task 05 — Read JSON orders: manual schema definition
# TODO: define the exact schema and re-read the file

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType
)

orders_schema = StructType([
    # TODO: define fields:
    # order_id (String), customer_id (String), product_id (String),
    # store_id (String), order_datetime (TimestampType),
    # quantity (IntegerType), unit_price (DoubleType),
    # discount_percent (DoubleType), total_amount (DoubleType),
    # payment_method (String), status (String)
])

df_orders = (
    spark.read
         # TODO: pass the schema and set timestampFormat "yyyy-MM-dd HH:mm:ss"
         .json(f"{DATASET_PATH}/orders/orders_batch.json")
)

print("=== JSON manual schema ===")
df_orders.printSchema()
df_orders.show(5, truncate=False)


In [ ]:
# Task 06 — Add audit metadata columns
# TODO: add two columns to df_orders:
#   _load_ts       = current ingestion timestamp   (use F.current_timestamp())
#   _source_file   = literal string with the file path  (use F.lit())

from pyspark.sql import functions as F

df_orders_bronze = (
    df_orders
    # TODO: .withColumn("_load_ts", ...)
    # TODO: .withColumn("_source_file", ...)
)

print("=== Bronze enriched schema ===")
df_orders_bronze.printSchema()
df_orders_bronze.select("order_id", "order_datetime", "_load_ts", "_source_file").show(5, truncate=False)


In [ ]:
# Task 07 — Write Bronze Delta table
# TODO: write df_orders_bronze to a managed Delta table called:
#   {CATALOG}.bronze.lab_orders_bronze
# Requirements:
#   - mode: overwrite
#   - format: delta
#   - create the bronze schema if it does not exist

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

(
    df_orders_bronze
    .write
    # TODO: set format to delta
    # TODO: set mode to overwrite
    # TODO: call .saveAsTable with the full 3-level name
)

print("Table written.")


In [ ]:
# Task 08 — Verify the table (GIVEN)

spark.sql(f"DESCRIBE DETAIL {CATALOG}.bronze.lab_orders_bronze").show(vertical=True)

row_count = spark.table(f"{CATALOG}.bronze.lab_orders_bronze").count()
print(f"Row count: {row_count}")

print("\n=== First 5 rows ===")
spark.table(f"{CATALOG}.bronze.lab_orders_bronze").show(5, truncate=False)


In [ ]:
# Task 09 — Append a second batch and verify idempotency
# A new batch of orders arrives. We append it to the existing table.
# TODO: read the stream file and APPEND it

BATCH2_PATH = f"{DATASET_PATH}/orders/stream/orders_stream_001.json"

df_batch2 = (
    spark.read
         # TODO: re-use orders_schema from Task 05
         # TODO: set timestampFormat
         .json(BATCH2_PATH)
         .withColumn("_load_ts", F.current_timestamp())
         .withColumn("_source_file", F.lit(BATCH2_PATH))
)

# TODO: write df_batch2 to the SAME table using mode="append"

print("After append:")
spark.table(f"{CATALOG}.bronze.lab_orders_bronze").groupBy("_source_file").count().show(truncate=False)


In [ ]:
# Task 10 — DataFrame transformations
# TODO: using df_orders (from Task 05) complete the following transformations

from pyspark.sql import functions as F

# 10a. Add a boolean column is_discounted = discount_percent > 0
df_t10 = df_orders.withColumn(
    "is_discounted",
    # TODO: condition using F.col
)

# 10b. Add revenue_net = total_amount * (1 - discount_percent / 100)
df_t10 = df_t10.withColumn(
    "revenue_net",
    # TODO: expression using F.col
)

# 10c. Group by payment_method → sum(revenue_net) aliased as total_revenue, count orders
df_summary = (
    df_t10
    .groupBy("payment_method")
    .agg(
        # TODO: F.sum("revenue_net").alias("total_revenue")
        # TODO: F.count("order_id").alias("order_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

df_summary.show(truncate=False)


In [ ]:
# Task 11 — inferSchema vs manual schema benchmark (GIVEN)
# Compare read time and schema accuracy

import time

path = f"{DATASET_PATH}/orders/orders_batch.json"

t0 = time.time()
spark.read.option("multiLine", "false").json(path).count()
infer_time = time.time() - t0

t0 = time.time()
spark.read.schema(orders_schema).json(path).count()
manual_time = time.time() - t0

print(f"inferSchema time : {infer_time:.2f} s")
print(f"manual schema time: {manual_time:.2f} s")
print(f"Speedup: {infer_time / manual_time:.1f}x faster with manual schema")

# ❓ The speedup is larger on the first run. Why?
# ❓ In production pipelines, which approach is recommended? Why?


In [ ]:
# Task 12 — Read CSV products → write as Delta
# TODO: fully implement: read products.csv → write to {CATALOG}.bronze.lab_products_bronze
# Requirements:
#   - Use a manual StructType schema (inspect the file first or use inferSchema to discover columns)
#   - Add _load_ts and _source_file metadata columns
#   - Write as Delta in overwrite mode to {CATALOG}.bronze.lab_products_bronze

# 12a — discovery (given)
spark.read.option("header","true").option("inferSchema","true")     .csv(f"{DATASET_PATH}/products/products.csv").printSchema()

# 12b — TODO: define schema, read, enrich, write
